In [108]:
import numpy as np
import cv2
from pathlib import Path
from scipy import fft
import matplotlib.pyplot as plt
from skimage import color
from tqdm import tqdm
import pickle
import torch


In [109]:
def resize_and_center_crop(image, target_size=448):
    """
    Resize image so smallest dimension equals target_size, then center crop to square.

    Args:
        image: Input image (numpy array)
        target_size: Target size for output square image (default 448)

    Returns:
        Cropped square image of size (target_size, target_size)
    """
    h, w = image.shape[:2]

    # Determine scale factor based on smallest dimension
    if h < w:
        # Height is smaller, scale based on height
        scale = target_size / h
        new_h = target_size
        new_w = int(w * scale)
    else:
        # Width is smaller, scale based on width
        scale = target_size / w
        new_w = target_size
        new_h = int(h * scale)

    # Resize image
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    # Center crop to target_size x target_size
    start_y = (new_h - target_size) // 2
    start_x = (new_w - target_size) // 2

    cropped = resized[start_y:start_y + target_size, start_x:start_x + target_size]

    return cropped


In [110]:
class CategorySignatureAnalyzer:
    def __init__(self, dataset_path, n_categories=565, n_exemplars=50):
        """
        Analyze spatial frequency, orientation, and chromaticity signatures per category.

        Args:
            dataset_path: Path to dataset with structure dataset_path/category_name/image_files
            n_categories: Number of categories (default 565)
            n_exemplars: Number of exemplars per category (default 50)
        """
        self.dataset_path = Path(dataset_path)
        self.n_categories = n_categories
        self.n_exemplars = n_exemplars
        self.results = {}

    def compute_fourier_signature(self, image):
        """
        Compute spatial frequency and orientation signature from image.

        Returns:
            freq_spectrum: 2D frequency spectrum (magnitude)
            orientation_hist: Histogram of orientations (36 bins, 10° each)
            radial_profile: Radial frequency profile
        """
        # Convert to grayscale if needed
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        else:
            gray = image

        # Compute 2D FFT
        ft = np.fft.ifftshift(gray)
        ft = np.fft.fft2(ft)
        ft = np.fft.fftshift(ft)

        return ft

    def compute_chromaticity_signature(self, image, n_samples=10000):
        """
        Compute CIELab chromaticity distribution.

        Args:
            image: RGB image
            n_samples: Number of pixels to randomly sample (default 10000)

        Returns:
            a_values: Array of a* values
            b_values: Array of b* values
        """
        # Ensure image is in correct format (0-1 range for skimage)
        if image.dtype == np.uint8:
            image = image.astype(np.float32) / 255.0

        # Convert to CIELab
        lab_image = color.rgb2lab(image)

        # Flatten spatial dimensions
        h, w = lab_image.shape[:2]
        lab_flat = lab_image.reshape(-1, 3)

        # Randomly sample pixels
        n_pixels = lab_flat.shape[0]
        if n_pixels > n_samples:
            indices = np.random.choice(n_pixels, size=n_samples, replace=False)
            lab_sample = lab_flat[indices]
        else:
            lab_sample = lab_flat

        L_values = lab_sample[:, 0]
        a_values = lab_sample[:, 1]
        b_values = lab_sample[:, 2]

        return L_values, a_values, b_values

    def process_category(self, category_path):
        """
        Process all images in a category and aggregate signatures.

        Returns:
            category_signature: Dictionary containing aggregated signatures
        """
        image_files = list(category_path.glob('*.jpg')) + \
                     list(category_path.glob('*.png')) + \
                     list(category_path.glob('*.JPEG'))

        # Limit to n_exemplars if specified
        if len(image_files) > self.n_exemplars:
            image_files = image_files[:self.n_exemplars]

        # Storage for aggregated results
        all_orientation_hists = []
        all_radial_profiles = []
        all_fts = []
        all_L_values = []
        all_a_values = []
        all_b_values = []

        for img_path in image_files:
            try:
                # Load image
                image = cv2.imread(str(img_path))
                if image is None:
                    continue
                if dataset == 'ecoTest': #for ecoTest
                    image =  resize_and_center_crop(image, 448)
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                # Compute Fourier signature
                ft = self.compute_fourier_signature(image)
                # ignore small values, likely noise
                all_fts.append(ft)
                #all_orientation_hists.append(orientation_hist)
                #all_radial_profiles.append(radial_profile)

                # Compute chromaticity signature
                L_vals, a_vals, b_vals = self.compute_chromaticity_signature(image)
                all_L_values.extend(L_vals)
                all_a_values.extend(a_vals)
                all_b_values.extend(b_vals)

            except Exception as e:
                print(f"Error processing {img_path}: {e}")
                continue

        # Aggregate results
        all_fts = [a for a in all_fts if a.shape == (448, 448)]
        category_signature = {

            'ft': np.array(all_fts).mean(0),
            #'orientation_hist_mean': np.mean(all_orientation_hists, axis=0),
            #'orientation_hist_std': np.std(all_orientation_hists, axis=0),
            #'radial_profile_mean': np.mean(all_radial_profiles, axis=0),
            #'radial_profile_std': np.std(all_radial_profiles, axis=0),
            'L_values': np.array(all_L_values),
            'a_values': np.array(all_a_values),
            'b_values': np.array(all_b_values),
            'n_images': len(all_orientation_hists)
        }

        return category_signature

    def analyze_dataset(self):
        """
        Analyze entire dataset and compute signatures for all categories.
        """
        category_dirs = sorted([d for d in self.dataset_path.iterdir() if d.is_dir()])

        print(f"Found {len(category_dirs)} categories")

        for category_dir in tqdm(category_dirs, desc="Processing categories"):
            category_name = category_dir.name
            category_signature = self.process_category(category_dir)
            self.results[category_name] = category_signature
            self.plot_category_signatures(category_name, f"figures/signatures/{dataset}/signatures_{category_name}.png")

        print(f"Processed {len(self.results)} categories successfully")
    def plot_category_signatures(self, category_name, save_path=None):
        """
        Plot the signatures for a specific category.
        Two panels: polar Fourier representation and colored chromaticity scatter.
        """
        if category_name not in self.results:
            print(f"Category {category_name} not found")
            return

        sig = self.results[category_name]

        fig = plt.figure(figsize=(8, 4))

        # ===== Panel 1: Polar Fourier Transform =====
        ax1 = plt.subplot(121)

        values = np.log(abs(sig['ft']))
        target_size = len(values)//2
        downsampled = cv2.resize(values, (target_size, target_size), interpolation=cv2.INTER_AREA)
        ax1.imshow(downsampled , cmap = 'grey')
        ax1.set_title('Fourier transform',
                      fontsize=12)
        ax1.axis('off')

        # ===== Panel 2: Colored Chromaticity Scatter =====
        ax2 = plt.subplot(122)

        # Subsample for visualization if too many points
        n_plot = min(50000, len(sig['a_values']))
        indices = np.random.choice(len(sig['a_values']), n_plot, replace=False)

        L_vals = sig['L_values'][indices]
        a_vals = sig['a_values'][indices]
        b_vals = sig['b_values'][indices]

        # Convert Lab to RGB for coloring the points
        # Create Lab array with L=50 (mid-lightness) for better color visibility
        lab_colors = np.zeros((len(a_vals), 3))
        lab_colors[:, 0] = L_vals  # L* = 50 (mid-lightness)
        lab_colors[:, 1] = a_vals
        lab_colors[:, 2] = b_vals

        # Convert to RGB
        rgb_colors = color.lab2rgb(lab_colors.reshape(-1, 1, 3)).reshape(-1, 3)

        # Clip to valid range and handle out-of-gamut colors
        rgb_colors = np.clip(rgb_colors, 0, 1)

        # Create scatter plot with actual colors
        ax2.scatter(a_vals, b_vals, c=rgb_colors, alpha=0.6, s=10, edgecolors='none')
        ax2.set_xlabel('a* (green ← → red)', fontsize=11)
        ax2.set_ylabel('b* (blue ← → yellow)', fontsize=11)
        ax2.set_title('CIELab Chromaticity Distribution',
                      fontsize=12)
        ax2.grid(alpha=0.3, linestyle='--', linewidth=0.5)
        ax2.axhline(0, color='k', linewidth=0.8, alpha=0.5)
        ax2.axvline(0, color='k', linewidth=0.8, alpha=0.5)

        # Set reasonable limits for Lab space
        ax2.set_xlim(-100, 100)
        ax2.set_ylim(-100, 100)
        ax2.set_aspect('equal')
        fig.suptitle(f'\n{category_name.split('_')[1]}', fontsize=14)

        fig.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
        #plt.show()
        plt.close()

    def save_results(self, output_path):
        """
        Save computed signatures to disk.
        """
        with open(output_path, 'wb') as f:
            pickle.dump(self.results, f)
        print(f"Results saved to {output_path}")

    def load_results(self, input_path):
        """
        Load previously computed signatures.
        """
        with open(input_path, 'rb') as f:
            self.results = pickle.load(f)
        print(f"Loaded results for {len(self.results)} categories")




In [111]:

if __name__ == "__main__":
    # Configuration
    dataset = 'genLOC2'
    if dataset == 'genLOC2':
        DATASET_PATH = "/home/alban/Documents/datasets/genLOC2/raw_v1"  # Change this
    elif dataset == 'ecoTest':
        DATASET_PATH = "/home/alban/Documents/datasets/ecoset/test"
    OUTPUT_PATH = f"results/signatures/category_signatures_{dataset}.pkl"

    # Initialize analyzer
    analyzer = CategorySignatureAnalyzer(
        dataset_path=DATASET_PATH,
        n_categories=10,
        n_exemplars=50
    )

    # Analyze dataset
    analyzer.analyze_dataset()

    # Save results
    analyzer.save_results(OUTPUT_PATH)

    # Example: plot signatures for first category
    category_names = list(analyzer.results.keys())
    if len(category_names) > 0:
        analyzer.plot_category_signatures(
            category_names[0],
            save_path=f"figures/signatures/signatures_{category_names[0]}.png")

Found 565 categories


Processing categories: 100%|██████████| 565/565 [25:51<00:00,  2.75s/it]


Processed 565 categories successfully
Results saved to results/signatures/category_signatures_genLOC2.pkl
